# Preprocess Sleep-EDF Dataset

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Optional
import numpy as np
import mne


DATA_ROOT = Path("./physionet.org/files/sleep-edfx/1.0.0/sleep-cassette")
OUT_ROOT = Path("./sleep_edf_preprocessed")

EPOCH_LENGTH = 30.0

# Sleep-EDF stage mapping according to U-Sleep scheme
ANNOT_TO_STAGE = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,   # merged into N3
    "Sleep stage R": 4,
}


def find_psg_hyp_pairs(root: Path) -> List[Tuple[Path, Path]]:
    pairs = []

    psg_files = sorted(root.rglob("*-PSG.edf"))
    hyp_files = sorted(root.rglob("*-Hypnogram.edf"))

    hyp_index = {}
    for hyp in hyp_files:
        base = hyp.name.replace("-Hypnogram.edf", "")  
        if len(base) == 0:
            continue
        core = base[:-1] 
        hyp_index.setdefault(core, []).append(hyp)

    for psg in psg_files:
        base_psg = psg.name.replace("-PSG.edf", "") 
        if len(base_psg) == 0:
            continue
        core = base_psg[:-1]

        candidates = hyp_index.get(core, [])
        if len(candidates) == 1:
            pairs.append((psg, candidates[0]))
        elif len(candidates) > 1:
            print(f"[WARN] Multiple hypnograms for core '{core}', using first: "
                  f"{[c.name for c in candidates]}")
            pairs.append((psg, candidates[0]))
        else:
            print(f"[WARN] No hypnogram found for PSG {psg.name} (core='{core}')")

    return pairs



def keep_only_eeg(raw: mne.io.Raw):
    desired = ["EEG Fpz-Cz", "EEG Pz-Oz"]

    present = [ch for ch in desired if ch in raw.ch_names]
    if len(present) == 0:
        raise RuntimeError(f"No Sleep-EDF EEG channels found in {raw.ch_names}")

    raw.pick_channels(present)


def robust_scale_clip(raw: mne.io.Raw, clip_mult=20.0):
    data = raw.get_data()

    for ci in range(data.shape[0]):
        x = data[ci]
        med = np.median(x)
        q25 = np.percentile(x, 25)
        q75 = np.percentile(x, 75)
        iqr = max(q75 - q25, 1e-6)

        x = (x - med) / iqr
        x = np.clip(x, -clip_mult, clip_mult)
        data[ci] = x

    raw._data = data


def attach_annotations(raw: mne.io.Raw, psg_path: Path, hyp_path: Path):
    has_stages = any(
        "Sleep stage" in desc for desc in raw.annotations.description
    )

    if has_stages:
        return

    annots = mne.read_annotations(hyp_path)
    raw.set_annotations(annots)


def make_epochs_and_labels(raw: mne.io.Raw):
    event_id = {k: v + 1 for k, v in ANNOT_TO_STAGE.items()}

    events, used_event_id = mne.events_from_annotations(raw, event_id=event_id)
    if len(events) == 0:
        raise RuntimeError("No valid sleep stage annotations found.")

    sfreq = raw.info["sfreq"]

    epochs = mne.Epochs(
        raw,
        events,
        event_id=used_event_id,
        tmin=0.0,
        tmax=EPOCH_LENGTH - 1.0 / sfreq,
        baseline=None,
        preload=True,
    )

    inv_map = {v: k for k, v in used_event_id.items()}
    labels = np.array([ANNOT_TO_STAGE[inv_map[e[2]]] for e in epochs.events])

    return epochs, labels


def process_record(psg_path: Path, hyp_path: Path, out_dir: Path):
    print(f"\n=== Processing {psg_path.name} ===")

    raw = mne.io.read_raw_edf(psg_path, preload=True)
    attach_annotations(raw, psg_path, hyp_path)
    keep_only_eeg(raw)
    raw.set_channel_types({ch: "eeg" for ch in raw.ch_names})
    robust_scale_clip(raw, clip_mult=20.0)
    epochs, labels = make_epochs_and_labels(raw)
    X = epochs.get_data()
    ch_names = epochs.ch_names
    sfreq = epochs.info["sfreq"]
    out_name = psg_path.stem.replace("-PSG", "") + "_eegonly_usleep.npz"
    out_path = out_dir / out_name

    np.savez(
        out_path,
        X=X.astype(np.float32),
        y=labels.astype(np.int64),
        ch_names=np.array(ch_names),
        sfreq=np.array([sfreq], dtype=np.float32),
    )

    print(f"Saved {out_path} — shape={X.shape}, n_ch={len(ch_names)}")


def main():
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

    pairs = find_psg_hyp_pairs(DATA_ROOT)
    print(f"Found {len(pairs)} PSG/Hypnogram pairs.")

    for psg_path, hyp_path in pairs:
        try:
            process_record(psg_path, hyp_path, OUT_ROOT)
        except Exception as e:
            print(f"[ERROR] {psg_path.name}: {e}")


if __name__ == "__main__":
    main()


# Preprocess HMC Dataset

In [ ]:
import os
from pathlib import Path
import numpy as np
import mne
import csv
 

BASE = Path("datasets/HMC/“./“")
REC_DIR = BASE / "recordings"
 
OUT_DIR = Path("preprocessed_data/HMC")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
EPOCH_SEC = 30.0
H_FREQ = 35.0
THR = 20.0
 
# 5-class mapping: W=0, N1=1, N2=2, N3=3 (merge N4->N3 if ever appears), REM/R=4
STAGE_TO_5 = {
    "W": 0,
    "N1": 1,
    "N2": 2,
    "N3": 3,
    "N4": 3,
    "R": 4,
    "REM": 4,
}
 
def robust_scale_continuous(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    med = np.median(x, axis=1, keepdims=True)
    q25 = np.quantile(x, 0.25, axis=1, keepdims=True)
    q75 = np.quantile(x, 0.75, axis=1, keepdims=True)
    iqr = np.maximum(q75 - q25, eps)
    return (x - med) / iqr
 
def norm_ch(s: str) -> str:
    s = s.upper().strip()
    for pref in ["EEG", "EOG", "EMG", "ECG"]:
        if s.startswith(pref):
            s = s[len(pref):].strip()
    return "".join(ch for ch in s if ch.isalnum())
 
def resolve_ch_name(raw: mne.io.Raw, target_norm: str):
    for ch in raw.ch_names:
        if norm_ch(ch) == target_norm:
            return ch
    return None
 
def pick_or_derive_c4m1_c3m2(raw_full: mne.io.Raw):
    c4m1 = resolve_ch_name(raw_full, "C4M1")
    c3m2 = resolve_ch_name(raw_full, "C3M2")
 
    if c4m1 and c3m2:
        r = raw_full.copy().pick([c4m1, c3m2])
        r.load_data()
        r.rename_channels({c4m1: "C4-M1", c3m2: "C3-M2"})
        r.set_channel_types({"C4-M1": "eeg", "C3-M2": "eeg"})
        return r, "CASE1_use_C4-M1_C3-M2"
 
    c4 = resolve_ch_name(raw_full, "C4")
    c3 = resolve_ch_name(raw_full, "C3")
    m1 = resolve_ch_name(raw_full, "M1")
    m2 = resolve_ch_name(raw_full, "M2")
 
    if c4 and m1 and c3 and m2:
        r = raw_full.copy().pick([c4, m1, c3, m2])
        r.load_data()
        data = r.get_data()
        idx = {name: i for i, name in enumerate(r.ch_names)}
 
        c4_m1 = data[idx[c4]] - data[idx[m1]]
        c3_m2 = data[idx[c3]] - data[idx[m2]]
 
        info = mne.create_info(
            ch_names=["C4-M1", "C3-M2"],
            sfreq=r.info["sfreq"],
            ch_types=["eeg", "eeg"],
        )
        rr = mne.io.RawArray(np.vstack([c4_m1, c3_m2]), info, verbose="ERROR")
        return rr, "CASE2_derive_(C4-M1, C3-M2)"
 
    raise RuntimeError(
        "Could not resolve C4/M1 and C3/M2. "
        f"Available (first 40): {raw_full.ch_names[:40]}"
    )
 
def read_hmc_labels_from_txt(txt_path: Path, epoch_sec: float = 30.0) -> np.ndarray:
    events = {}
    with open(txt_path, "r", newline="") as f:
        reader = csv.reader(f, delimiter=",")
        for row in reader:
            if not row:
                continue
            if row[0].strip().lower() == "date":
                continue
            if len(row) < 5:
                continue
 
            onset_s = row[2].strip()
            ann = row[4].strip()
 
            if not ann.lower().startswith("sleep stage"):
                continue
            stage = ann.split()[-1].upper()
 
            if stage not in STAGE_TO_5:
                continue
 
            try:
                onset = float(onset_s)
            except Exception:
                continue
 
            ep_idx = int(round(onset / epoch_sec))
            events[ep_idx] = STAGE_TO_5[stage]
 
    if not events:
        raise RuntimeError(f"No sleep-stage rows parsed from: {txt_path.name}")
 
    max_idx = max(events.keys())
    y = np.full(max_idx + 1, -1, dtype=np.int64)
    for k, v in events.items():
        y[k] = v
 
    return y
 
def bincount5(y):
    return np.bincount(y, minlength=5)
 
def list_recordings(rec_dir: Path):
    edfs = sorted(rec_dir.glob("SN*.edf"))
    edfs = [p for p in edfs if not p.name.endswith("_sleepscoring.edf")]
    return edfs
 

def process_recording(edf_path: Path):
    sid = edf_path.stem
    txt_path = edf_path.with_name(f"{sid}_sleepscoring.txt")
 
    if not txt_path.exists():
        raise FileNotFoundError(f"Missing labels: {txt_path}")
 
    raw_full = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")
 
    raw, case = pick_or_derive_c4m1_c3m2(raw_full)
    sfreq = float(raw.info["sfreq"])
 
    raw.filter(l_freq=None, h_freq=H_FREQ, picks="all", verbose="ERROR")
 
    x = robust_scale_continuous(raw.get_data())  # (2, T)
 
    n_samp_ep = int(round(EPOCH_SEC * sfreq))
    n_epochs_sig = x.shape[1] // n_samp_ep
    x = x[:, : n_epochs_sig * n_samp_ep]
    X = x.reshape(2, n_epochs_sig, n_samp_ep).transpose(1, 0, 2)  # (E,2,T)
 
    y = read_hmc_labels_from_txt(txt_path, epoch_sec=EPOCH_SEC)  # (E_label,)
    n = min(len(y), X.shape[0])
    X = X[:n]
    y = y[:n]
 
    valid = (y != -1)
    X = X[valid]
    y = y[valid]
 
    max_abs = np.max(np.abs(X), axis=(1, 2))
    bad = max_abs > THR
    X_keep = X[~bad]
    y_keep = y[~bad]
 
    removed = int(bad.sum())
    total = int(len(bad))
    pct_removed = 100.0 * removed / max(total, 1)
 
    out_path = OUT_DIR / f"HMC_{sid}_C4M1_C3M2_lp{int(H_FREQ)}_reject_thr{THR:g}.npz"
    np.savez_compressed(
        out_path,
        X=X_keep.astype(np.float32),
        y=y_keep.astype(np.int64),
        ch=np.array(["C4-M1", "C3-M2"]),
        sfreq=np.array([sfreq], dtype=np.float32),
        epoch_sec=np.array([EPOCH_SEC], dtype=np.float32),
        h_freq=np.array([H_FREQ], dtype=np.float32),
        thr=np.array([THR], dtype=np.float32),
        removed=np.array([removed], dtype=np.int64),
        total=np.array([total], dtype=np.int64),
        channel_case=np.array([case]),
    )
 
    return {
        "sid": sid,
        "case": case,
        "sfreq": sfreq,
        "epochs_total": total,
        "epochs_removed": removed,
        "pct_removed": pct_removed,
        "dist_before": bincount5(y).tolist() if len(y) else [0,0,0,0,0],
        "dist_after": bincount5(y_keep).tolist() if len(y_keep) else [0,0,0,0,0],
        "out": str(out_path),
    }
 

edfs = list_recordings(REC_DIR)
print("Found EDF recordings:", len(edfs))
 
results, fails = [], []
case_counts = {}
 
for edf_path in edfs:
    sid = edf_path.stem
    try:
        r = process_recording(edf_path)
        results.append(r)
        case_counts[r["case"]] = case_counts.get(r["case"], 0) + 1
 
        print(
            f"{sid} | {r['case']} | sfreq={r['sfreq']:.1f} | "
            f"removed {r['epochs_removed']}/{r['epochs_total']} ({r['pct_removed']:.2f}%) | "
            f"before={r['dist_before']} | after={r['dist_after']}"
        )
    except Exception as e:
        fails.append((sid, str(e)))
        print(f"[FAIL] {sid}: {e}")
 
print("\nDone. Processed:", len(results), "| Failed:", len(fails))
print("\nChannel-case counts:")
for k, v in sorted(case_counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {k}: {v}")

Found EDF recordings: 151
Reading 0 ... 6566399  =      0.000 ... 25649.996 secs...
SN001 | CASE1_use_C4-M1_C3-M2 | sfreq=256.0 | removed 59/854 (6.91%) | before=[151, 109, 430, 23, 141] | after=[123, 94, 419, 23, 136]
Reading 0 ... 6577407  =      0.000 ... 25692.996 secs...
SN002 | CASE1_use_C4-M1_C3-M2 | sfreq=256.0 | removed 35/856 (4.09%) | before=[82, 67, 309, 224, 174] | after=[65, 65, 297, 222, 172]
Reading 0 ... 7330815  =      0.000 ... 28635.996 secs...
SN003 | CASE1_use_C4-M1_C3-M2 | sfreq=256.0 | removed 120/954 (12.58%) | before=[403, 81, 168, 150, 152] | after=[294, 74, 167, 149, 150]
Reading 0 ... 7804927  =      0.000 ... 30487.996 secs...
SN004 | CASE1_use_C4-M1_C3-M2 | sfreq=256.0 | removed 31/1016 (3.05%) | before=[80, 91, 534, 64, 247] | after=[66, 85, 528, 64, 242]
Reading 0 ... 7370751  =      0.000 ... 28791.996 secs...
SN005 | CASE1_use_C4-M1_C3-M2 | sfreq=256.0 | removed 89/959 (9.28%) | before=[201, 141, 341, 149, 127] | after=[147, 126, 329, 146, 122]
Readin